# Lecture 18: Productionizing NLP Systems — Answer Key
### NLP Course 2027 — Week 09

---
Complete answers for all four exercises in Lecture 18.

In [ ]:
import time
import json
import numpy as np
import matplotlib.pyplot as plt
from functools import lru_cache
print('Ready')

---
## Exercise 18.1 — FastAPI NLP Service

**Task:** Build a FastAPI service with `/sentiment`, `/ner`, and `/summarize` endpoints.

**Key concept:** Production NLP APIs should load models once at startup (not per request), validate input (length limits, type checks), log every request (for monitoring), and handle errors gracefully (HTTP 400/500 with informative messages).

In [ ]:
fastapi_app = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, validator
from transformers import pipeline
import time
import logging

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

app = FastAPI(title="NLP API", version="1.0.0")

# Models loaded once at startup
models = {}

class TextRequest(BaseModel):
    text: str

    @validator("text")
    def text_not_empty(cls, v):
        if not v.strip():
            raise ValueError("text must not be empty")
        return v

@app.on_event("startup")
async def load_models():
    logger.info("Loading models...")
    models["sentiment"] = pipeline(
        "text-classification",
        model="distilbert-base-uncased-finetuned-sst-2-english")
    models["ner"] = pipeline(
        "ner",
        model="dbmdz/bert-large-cased-finetuned-conll03-english",
        aggregation_strategy="simple")
    models["summarize"] = pipeline(
        "summarization", model="facebook/bart-large-cnn")
    logger.info("All models loaded.")

@app.post("/sentiment")
async def sentiment(req: TextRequest):
    if len(req.text) > 10_000:
        raise HTTPException(400, "Text too long (max 10,000 chars)")
    t0 = time.time()
    result = models["sentiment"](req.text)[0]
    logger.info(f"/sentiment latency={1000*(time.time()-t0):.1f}ms")
    return {"label": result["label"], "score": round(result["score"], 4)}

@app.post("/ner")
async def ner(req: TextRequest):
    if len(req.text) > 5_000:
        raise HTTPException(400, "Text too long (max 5,000 chars)")
    t0 = time.time()
    entities = models["ner"](req.text)
    logger.info(f"/ner latency={1000*(time.time()-t0):.1f}ms")
    return {"entities": [{"text": e["word"], "type": e["entity_group"],
                          "score": round(e["score"], 4)} for e in entities]}

@app.post("/summarize")
async def summarize(req: TextRequest):
    words = req.text.split()
    if len(words) < 50:
        raise HTTPException(400, "Text too short to summarize (min 50 words)")
    t0 = time.time()
    result = models["summarize"](req.text, max_length=130, min_length=30, do_sample=False)
    logger.info(f"/summarize latency={1000*(time.time()-t0):.1f}ms")
    return {"summary": result[0]["summary_text"]}

@app.get("/health")
async def health():
    return {"status": "healthy", "models_loaded": list(models.keys())}

# Run: uvicorn nlp_api:app --host 0.0.0.0 --port 8000
'''

with open('/tmp/nlp_api.py', 'w') as f:
    f.write(fastapi_app)

print('FastAPI service written to /tmp/nlp_api.py')
print('\nKey design decisions:')
design = [
    'Models loaded once at startup (not per request) — avoids 2–5s cold start per call',
    'Pydantic validators catch empty/None input before it reaches the model',
    'Length limits prevent OOM errors on unexpectedly long inputs',
    'Logging captures latency for every request — essential for SLO monitoring',
    'HTTP 400 for client errors (bad input), 500 for server errors (model crash)',
    '/health endpoint enables load balancer health checks and Kubernetes liveness probes',
]
for d in design:
    print(f'  • {d}')

---
## Exercise 18.2 — Latency Benchmarking

**Task:** Measure avg/p50/p95/p99 latency and throughput.

**Key concept:** SLO (Service Level Objective) targets typically focus on p95 latency, not average. A model with avg=50ms but p99=2000ms will fail 1% of requests — unacceptable for high-traffic services. Always measure percentiles.

In [ ]:
try:
    from transformers import pipeline
    sentiment = pipeline('text-classification',
                         model='distilbert-base-uncased-finetuned-sst-2-english')
    model_available = True
except Exception:
    model_available = False

def benchmark_model(model_fn, texts, n_runs=3):
    latencies = []
    for _ in range(n_runs):
        for text in texts:
            t0 = time.perf_counter()
            model_fn(text)
            latencies.append((time.perf_counter() - t0) * 1000)  # ms
    latencies = sorted(latencies)
    n = len(latencies)
    return {
        'n_requests':   n,
        'avg_ms':       np.mean(latencies),
        'min_ms':       latencies[0],
        'p50_ms':       latencies[int(n * 0.50)],
        'p95_ms':       latencies[int(n * 0.95)],
        'p99_ms':       latencies[int(n * 0.99)],
        'max_ms':       latencies[-1],
        'throughput_rps': 1000 / np.mean(latencies),
    }

texts = [
    'This product is excellent!',
    'Terrible experience, would not recommend.',
    'Average quality, nothing special.',
    'Absolutely love this, best purchase ever.',
    'Very disappointed, does not work as described.',
] * 10  # 50 texts

if model_available:
    results = benchmark_model(lambda t: sentiment(t), texts)
    print('Benchmark Results:')
    for k, v in results.items():
        unit = ' ms' if 'ms' in k else (' req/s' if 'throughput' in k else '')
        print(f'  {k:<20}: {v:.2f}{unit}')
    print()
    print('SLO guidance: if p95 > 200ms for a user-facing API, consider batching or GPU.')
else:
    # Simulate results for illustration
    np.random.seed(42)
    sim_latencies = np.abs(np.random.normal(45, 15, 150))  # ~45ms avg
    sim_latencies = sorted(sim_latencies)
    n = len(sim_latencies)
    print('Simulated benchmark (model not available):')
    print(f'  avg_ms:    {np.mean(sim_latencies):.2f} ms')
    print(f'  p50_ms:    {sim_latencies[int(n*0.50)]:.2f} ms')
    print(f'  p95_ms:    {sim_latencies[int(n*0.95)]:.2f} ms')
    print(f'  p99_ms:    {sim_latencies[int(n*0.99)]:.2f} ms')
    print(f'  throughput: {1000/np.mean(sim_latencies):.1f} req/s')

---
## Exercise 18.3 — Prediction Caching

**Task:** Implement `PredictionCache`. Simulate varying repetition rates and plot hit rate.

**Key concept:** Caching is highly effective for NLP APIs where many users ask the same questions (FAQ bots, search). Hit rate grows rapidly with repetition rate. A 50% repetition rate → ~50% hit rate → 2× throughput improvement.

In [ ]:
class PredictionCache:
    """LRU-style prediction cache with hit/miss tracking."""

    def __init__(self, model_fn, max_size=500):
        self.model_fn  = model_fn
        self.max_size  = max_size
        self._cache    = {}       # ordered dict via insertion order (Python 3.7+)
        self.hits      = 0
        self.misses    = 0
        self.total_time_saved_ms = 0.0

    def predict(self, text, avg_model_ms=45.0):
        if text in self._cache:
            self.hits += 1
            self.total_time_saved_ms += avg_model_ms
            return self._cache[text]
        result = self.model_fn(text)
        if len(self._cache) >= self.max_size:
            self._cache.pop(next(iter(self._cache)))  # evict oldest
        self._cache[text] = result
        self.misses += 1
        return result

    def stats(self):
        total = self.hits + self.misses
        return {
            'hits': self.hits, 'misses': self.misses, 'total': total,
            'hit_rate': self.hits / total if total > 0 else 0,
            'time_saved_ms': self.total_time_saved_ms,
        }

    def reset(self):
        self._cache.clear(); self.hits = 0; self.misses = 0; self.total_time_saved_ms = 0


# Simulate traffic at different repetition rates
def fake_model(text): return {'label': 'POSITIVE', 'score': 0.99}  # dummy

unique_queries = [f'Is product {i} good?' for i in range(100)]
repetition_rates = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
observed_hit_rates = []

cache = PredictionCache(fake_model, max_size=500)

np.random.seed(42)
for rep_rate in repetition_rates:
    cache.reset()
    # Generate 1000 requests: rep_rate fraction are repeated queries
    for _ in range(1000):
        if np.random.rand() < rep_rate and len(unique_queries) > 0:
            text = np.random.choice(unique_queries[:20])  # repeated from top-20
        else:
            text = np.random.choice(unique_queries)
        cache.predict(text)
    s = cache.stats()
    observed_hit_rates.append(s['hit_rate'])
    print(f'Repetition rate={rep_rate:.0%}: hit_rate={s["hit_rate"]:.3f}, '
          f'time_saved={s["time_saved_ms"]/1000:.1f}s')

plt.figure(figsize=(7, 4))
plt.plot(repetition_rates, observed_hit_rates, marker='o', color='steelblue')
plt.xlabel('Traffic Repetition Rate'); plt.ylabel('Cache Hit Rate')
plt.title('Cache Hit Rate vs Traffic Repetition Rate')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## Exercise 18.4 — MLflow Experiment Tracking

**Task:** Set up MLflow to track fine-tuning hyperparameters and metrics.

**Key concept:** Experiment tracking records every hyperparameter, metric, and artifact so you can reproduce any run, compare configurations, and deploy the best model. MLflow stores runs locally (./mlruns) or on a remote server.

In [ ]:
mlflow_code = '''
import mlflow
import mlflow.pytorch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding
)
import evaluate

mlflow.set_tracking_uri("./mlruns")  # local directory
mlflow.set_experiment("nlp-distilbert-sentiment")

MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
metric     = evaluate.load("accuracy")

dataset = load_dataset("sst2")
def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=128)
tokenized = dataset.map(tokenize, batched=True, remove_columns=["sentence", "idx"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return metric.compute(predictions=np.argmax(logits, -1), references=labels)

hyperparams = [
    {"lr": 2e-5, "epochs": 3, "batch_size": 32},
    {"lr": 5e-5, "epochs": 3, "batch_size": 32},
    {"lr": 2e-5, "epochs": 5, "batch_size": 16},
]

for hp in hyperparams:
    run_name = f"lr={hp[\'lr\']}_ep={hp[\'epochs\']}_bs={hp[\'batch_size\']}"
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(hp)

        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
        args  = TrainingArguments(
            output_dir=f"./results/{run_name}",
            learning_rate=hp["lr"],
            num_train_epochs=hp["epochs"],
            per_device_train_batch_size=hp["batch_size"],
            evaluation_strategy="epoch",
            report_to="none",
        )
        trainer = Trainer(
            model=model, args=args,
            train_dataset=tokenized["train"],
            eval_dataset=tokenized["validation"],
            tokenizer=tokenizer,
            data_collator=DataCollatorWithPadding(tokenizer),
            compute_metrics=compute_metrics,
        )
        trainer.train()
        results = trainer.evaluate()

        # Log all eval metrics
        for k, v in results.items():
            if isinstance(v, (int, float)):
                mlflow.log_metric(k.replace("eval_", ""), v)

        # Save model artifact
        mlflow.pytorch.log_model(model, "model")
        print(f"{run_name}: accuracy={results.get(\'eval_accuracy\', 0):.4f}")

print("All runs complete. View at: mlflow ui --port 5000")
'''

print('MLflow tracking code:')
print(mlflow_code)

print('\nKey MLflow concepts:')
concepts = [
    'mlflow.log_params(dict)  — hyperparameters (lr, batch_size, etc.)',
    'mlflow.log_metric(k, v)  — scalar metrics per epoch (accuracy, loss)',
    'mlflow.log_artifact(path) — save files (confusion matrix plots, config)',
    'mlflow.pytorch.log_model — save the full model for deployment',
    'mlflow ui                 — browser UI to compare all runs side by side',
]
for c in concepts:
    print(f'  {c}')

---
## Summary

| Exercise | Key Concept |
|----------|-------------|
| 18.1 | Load models at startup; validate input; log latency; use HTTP 400/500 appropriately |
| 18.2 | Always measure p95/p99 latency, not just average — tail latency drives SLO violations |
| 18.3 | Cache hit rate grows with traffic repetition; 50% repeat → ~2× throughput |
| 18.4 | MLflow: log params + metrics + model artifacts → reproducible experiments |

---
*NLP Course 2027 — Week 09*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**